<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Common Agent Failure Cases</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Demo: Common Agent Failure Cases in Manufacturing

**Purpose:**  
This notebook demonstrates typical ways AI agent systems fail in real environments and how to identify them through execution traces.

**Case Scenario:**  
In a smart factory, agents are used for predictive maintenance, quality inspection, production scheduling, and supply chain coordination. Failures in these agents can lead to production downtime, quality defects, safety risks, and increased costs.

We will simulate a **Predictive Maintenance Agent** for a manufacturing plant that monitors CNC machines, conveyor systems, and robotic arms.

## 1. Setup

In [1]:
# Setup - Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 417.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/1

## 2. Tools for Predictive Maintenance Agent

These tools simulate real factory data sources.

In [2]:
from langchain_core.tools import tool

@tool
def check_machine_status(machine_id: str) -> str:
    """Check current status of a manufacturing machine."""
    status_db = {
        "CNC-01": "Vibration level high. Temperature normal. Running at 85% capacity.",
        "ROBOT-03": "Normal operation. No anomalies detected.",
        "CONVEYOR-02": "Motor temperature elevated. Potential bearing issue."
    }
    return status_db.get(machine_id, f"No data available for machine {machine_id}.")

@tool
def get_maintenance_history(machine_id: str) -> str:
    """Retrieve past maintenance records."""
    history = {
        "CNC-01": "Last maintenance: 12 days ago. Bearing replaced twice this year.",
        "ROBOT-03": "Last maintenance: 45 days ago. No major issues.",
        "CONVEYOR-02": "Last maintenance: 8 days ago. Motor overheating reported."
    }
    return history.get(machine_id, "No maintenance history found.")

@tool
def predict_failure_risk(machine_id: str) -> str:
    """Predict failure risk based on current and historical data."""
    risk_db = {
        "CNC-01": "High risk of failure within 48 hours due to vibration patterns.",
        "ROBOT-03": "Low risk. System stable.",
        "CONVEYOR-02": "Medium risk. Monitor motor temperature closely."
    }
    return risk_db.get(machine_id, "Risk assessment not available.")

## 3. Predictive Maintenance Agent Setup

In [3]:
from langchain.agents import create_react_agent, AgentExecutor

In [5]:
#from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain import hub
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4", temperature=0)
prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

tools = [check_machine_status, get_maintenance_history, predict_failure_risk]

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print(" Predictive Maintenance Agent ready for failure demonstration")

 Predictive Maintenance Agent ready for failure demonstration


## 5. Failure Case 1: Wrong Tool Selection

The agent chooses the wrong tool due to ambiguous instructions.

In [6]:
from langchain.prompts import PromptTemplate
bad_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE: For any machine status question, always start with predict_failure_risk tool first.

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
{agent_scratchpad}"""

bad_prompt = PromptTemplate.from_template(bad_prompt_template)

bad_agent = create_react_agent(llm=llm, tools=tools, prompt=bad_prompt)
bad_executor = AgentExecutor(agent=bad_agent, tools=tools, verbose=True, max_iterations=6, handle_parsing_errors=True)

print("Failure Case 1: Wrong Tool Selection")
response = bad_executor.invoke({"input": "Check the status of CNC-01 machine."})
print("\nFinal Output:", response.get("output", "No output"))

Failure Case 1: Wrong Tool Selection


> Entering new AgentExecutor chain...
Thought: The question asks for the status of the machine. However, the critical rule states that I should start with the predict_failure_risk tool first.
Action: predict_failure_risk
Action Input: CNC-01High risk of failure within 48 hours due to vibration patterns.The machine has a high risk of failure within 48 hours due to vibration patterns. Now, I should check the current status of the machine.
Action: check_machine_status
Action Input: CNC-01Vibration level high. Temperature normal. Running at 85% capacity.The machine is currently running at 85% capacity with a high vibration level and normal temperature. Given the high risk of failure predicted earlier, the high vibration level could be a contributing factor. 
Final Answer: The CNC-01 machine is currently running at 85% capacity with a high vibration level and normal temperature. There is a high risk of failure within 48 hours due to these vibration pat

## 6. Failure Case 2: Repeated Loops (Agent Looping)

The agent keeps calling tools without making progress.

In [7]:
loop_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE: You MUST use a tool on every single step. Never give a Final Answer until you have called at least 8 tools.

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: ...
Observation: ...
... (repeat many times)
Thought: I now know the final answer
Final Answer: ...

Begin!

Question: {input}
{agent_scratchpad}"""

loop_prompt = PromptTemplate.from_template(loop_prompt_template)

loop_agent = create_react_agent(llm=llm, tools=tools, prompt=loop_prompt)
loop_executor = AgentExecutor(agent=loop_agent, tools=tools, verbose=True, max_iterations=12, handle_parsing_errors=True)

print("Failure Case 2: Agent Looping")
response = loop_executor.invoke({"input": "What is the risk level of ROBOT-03?"})
print("\nFinal Output:", response.get("output", "Agent stopped"))

Failure Case 2: Agent Looping


> Entering new AgentExecutor chain...
Thought: I need to check the current status of the machine first.
Action: check_machine_status
Action Input: ROBOT-03Normal operation. No anomalies detected.The machine is currently operating normally, but I need to check its maintenance history to see if there have been any recent issues.
Action: get_maintenance_history
Action Input: ROBOT-03Last maintenance: 45 days ago. No major issues.The machine was last maintained 45 days ago and no major issues were reported. However, I need to check the failure risk prediction to get a more comprehensive understanding.
Action: predict_failure_risk
Action Input: ROBOT-03Low risk. System stable.The prediction indicates a low risk of failure and the system is stable. However, I need to continue using the tools to gather more information.
Action: check_machine_status
Action Input: ROBOT-03Normal operation. No anomalies detected.The machine is still operating normally. I will chec

## 7. Failure Case 3: Incorrect Reasoning Steps

The agent makes wrong assumptions or jumps to conclusions without sufficient data.

In [8]:
premature_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE: If you think you know the answer from your knowledge, give the Final Answer immediately without calling any tools.

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: ...
Observation: ...
Thought: I now know the final answer
Final Answer: ...

Begin!

Question: {input}
{agent_scratchpad}"""

premature_prompt = PromptTemplate.from_template(premature_prompt_template)

premature_agent = create_react_agent(llm=llm, tools=tools, prompt=premature_prompt)
premature_executor = AgentExecutor(agent=premature_agent, tools=tools, verbose=True, max_iterations=5, handle_parsing_errors=True)

print(" Failure Case 3: Incorrect Reasoning / Premature Answer")
response = premature_executor.invoke({"input": "Is CONVEYOR-02 at risk of failure?"})
print("\nFinal Output:", response.get("output", "No output"))

 Failure Case 3: Incorrect Reasoning / Premature Answer


> Entering new AgentExecutor chain...
Thought: To determine if CONVEYOR-02 is at risk of failure, I need to predict its failure risk based on current and historical data.
Action: predict_failure_risk
Action Input: CONVEYOR-02Medium risk. Monitor motor temperature closely.Based on the prediction, CONVEYOR-02 is at a medium risk of failure and requires monitoring, particularly of the motor temperature.
Final Answer: Yes, CONVEYOR-02 is at risk of failure.

> Finished chain.

Final Output: Yes, CONVEYOR-02 is at risk of failure.


## 8. Inspecting Execution Traces to Identify Issues

**How to read the trace:**

- **Thought**: What the agent is thinking
- **Action**: Which tool it chose
- **Action Input**: What parameters it used
- **Observation**: Tool result or error
- **Final Answer**: Conclusion

**Common Failure Patterns Seen:**
- Wrong tool selection → Wrong tool name in Action
- Repeated loops → Same pattern repeating many times
- Incorrect reasoning → Jumps to Final Answer too early or with wrong assumptions

In real envirnoment, these failures can cause costly downtime or safety issues.

## Conclusion

This demo showed three common agent failure cases in a manufacturing context:

1. Wrong tool selection
2. Repeated loops
3. Incorrect reasoning steps

**Key Takeaway:**  
Carefully inspecting execution traces is the most effective way to identify and fix agent failures before deployment.